In [ ]:
from openai import OpenAI
from datetime import datetime
import os
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 함수 정의
def get_current_time():
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print("Current time:", now)
    return now

# 도구 정의
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "현재 날짜와 시간을 반환합니다."
        }
    }
]

# AI 호출 함수
def get_ai_response(messages):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    return response

if __name__ == "__main__":
    messages = [
        {"role": "user", "content": "지금 시간이 몇 시야?"}
    ]

    response = get_ai_response(messages)
    message = response.choices[0].message

    # 도구 호출 감지
    if message.tool_calls:
        tool_name = message.tool_calls[0].function.name

        if tool_name == "get_current_time":
            result = get_current_time()

            # 함수 실행 결과를 모델에 다시 전달
            messages.append({
                "role": "tool",
                "name": tool_name,
                "content": result
            })

            # 모델이 최종 응답 생성
            final_response = get_ai_response(messages)
            print("AI:", final_response.choices[0].message.content)
    else:
        print("AI:", message.content)

Current time: 2025-10-26 19:22:07


In [ ]:
from openai import OpenAI
from datetime import datetime
from dotenv import load_dotenv
import os

# 환경 변수 로드
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 도구로 쓸 함수
def get_current_time():
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print("Current time:", now)
    return now

# 함수 정의를 모델에 전달
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "현재 날짜와 시간을 반환합니다."
        }
    }
]

# 모델 호출 함수
def get_ai_response(messages, tools=None):
    return client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

# 메인 실행
if __name__ == "__main__":
    messages = [
        {"role": "system", "content": "너는 사용자를 도와주는 상담사야."}
    ]

    while True:
        user_input = input("사용자:\t")
        if user_input.lower() in ["exit", "quit"]:
            break

        messages.append({"role": "user", "content": user_input})
        response = get_ai_response(messages, tools=tools)
        message = response.choices[0].message

        # 도구 호출 감지
        if message.tool_calls:
            tool_name = message.tool_calls[0].function.name
            tool_call_id = message.tool_calls[0].id

            if tool_name == "get_current_time":
                tool_result = get_current_time()

                # 도구 실행 결과를 모델에 다시 전달
                messages.append({
                    "role": "function",
                    "name": tool_name,
                    "content": tool_result,
                    "tool_call_id": tool_call_id
                })

                # 모델이 최종 답변 생성
                final_response = get_ai_response(messages, tools=tools)
                ai_message = final_response.choices[0].message
                print("AI:\t", ai_message.content)
                messages.append(ai_message)
        else:
            print("AI:\t", message.content)
            messages.append(message)

Current time: 2025-10-26 19:30:52
AI:	 지금 시간은 2025년 10월 26일 19시 30분 52초입니다.


In [1]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., description="주식 코드 (예: 'AAPL')")
    period: str = Field(..., title="기간", description="주식 데이터를 조회할 기간 (예: '1mo', '3mo', '1y')")
  